In [0]:
# generer les evenements 
from pyspark.sql.functions import (
    col,
    expr,
    floor,
    from_unixtime,
    lit,
    rand,
    when
)

EVENT_COUNT = 1000

incoming_path = (
    "/Volumes/retail_dev/ops/"
    "landing_volume/stock_events/incoming"
)

events_df = (
    spark.range(EVENT_COUNT)

    .withColumn("event_id", expr("uuid()"))

    .withColumn(
        "stock_item_id",
        (floor(rand(42) * 227) + 1).cast("int")
    )

    .withColumn(
        "store_id",
        (floor(rand(43) * 10) + 1).cast("int")
    )

    .withColumn("event_selector", rand(44))

    .withColumn(
        "event_type",
        when(col("event_selector") < 0.65, "SALE")
        .when(col("event_selector") < 0.80, "RESTOCK")
        .when(col("event_selector") < 0.90, "RETURN")
        .otherwise("ADJUSTMENT")
    )

    .withColumn(
        "quantity_change",
        when(
            col("event_type") == "SALE",
            -(floor(rand(45) * 5) + 1)
        )
        .when(
            col("event_type") == "RESTOCK",
            floor(rand(45) * 50) + 10
        )
        .when(
            col("event_type") == "RETURN",
            floor(rand(45) * 3) + 1
        )
        .otherwise(
            floor(rand(45) * 11) - 5
        )
        .cast("int")
    )

    .withColumn(
        "event_timestamp",
        from_unixtime(
            expr(
                "unix_timestamp() "
                "- CAST(rand(46) * 2592000 AS BIGINT)"
            )
        ).cast("timestamp")
    )

    .withColumn(
        "source_system",
        lit("store_pos_simulator")
    )

    .select(
        "event_id",
        "stock_item_id",
        "store_id",
        "event_type",
        "quantity_change",
        "event_timestamp",
        "source_system"
    )
)

events_df.write.mode("append").json(incoming_path)

print(f"{EVENT_COUNT} événements déposés dans {incoming_path}")

display(events_df.limit(10))

In [0]:
# Ingestion avec Auto Loader
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType
)

stock_event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("stock_item_id", IntegerType(), False),
    StructField("store_id", IntegerType(), False),
    StructField("event_type", StringType(), False),
    StructField("quantity_change", IntegerType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False)
])

schema_path = (
    "/Volumes/retail_dev/ops/"
    "technical_volume/autoloader/stock_events/schema"
)

checkpoint_path = (
    "/Volumes/retail_dev/ops/"
    "technical_volume/checkpoints/stock_events"
)

stock_events_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .schema(stock_event_schema)
    .load(incoming_path)

    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query = (
    stock_events_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("retail_dev.bronze.stock_events")
)

query.awaitTermination()

print("Ingestion Auto Loader terminée.")

In [0]:
# Verification 
stock_events_bronze = spark.table(
    "retail_dev.bronze.stock_events"
)

print(
    f"Nombre d'événements Bronze : "
    f"{stock_events_bronze.count()}"
)

display(
    stock_events_bronze
    .orderBy(col("event_timestamp").desc())
    .limit(20)
)